# Surface Heat Flow Prediction — Training & Hyperparameter Tuning


In [1]:
# Install MultKAN dependencies
import sys, subprocess
try:
    import yaml
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyyaml', '-q'])
    import yaml
print(f"✅ yaml OK, sys.path includes pykan: {'pykan-master' in str(sys.path)}")


✅ yaml OK, sys.path includes pykan: False


## Section 1: Import Required Libraries and Configuration


In [2]:
from models import BSRBF_KAN, EfficientKAN, FasterKAN, MLP, FastKAN, GottliebKAN, RBFKAN
from file_io import write_single_dict_to_jsonl_file


## Section 2: ⚙️ Hyperparameter Configuration (modify parameters here!)

Change the values in the dictionary below, then re-run the following cells to try different configurations.


In [3]:
# ====== Cell C: MLP ======
class Args:
    data_file = r'c:\KAN-OLS-SHF\output\model\mesh\shf_with_distance_vectors.pkl'
    output_dir = r'c:\KAN-OLS-SHF\output\train\mlp'
    model_type = 'mlp'
    n_hidden = 64
    num_rbf = 5
    grid_size = 15
    spline_order = 5
    epochs = 200
    lr = 1e-3
    weight_decay = 1e-3
    batch_size = 32
    pca_components = 20
    cv_folds = 10
    seed = 86
    scheduler_type = 'plateau'
    huber_delta = 1.0
    regressor = 'ols'
    ridge_alpha = 10.0
args = Args()
print(f"🔬 MLP: hidden={args.n_hidden}, α={args.ridge_alpha}, plateau")


🔬 MLP: hidden=64, α=10.0, plateau


In [4]:
# Test RBFKAN import
try:
    from models.rbf_kan import RBFKAN
    m = RBFKAN([20, 64, 64, 12], num_rbf=5)
    print(f"✅ RBFKAN 导入成功, params={sum(p.numel() for p in m.parameters()):,}")
except Exception as e:
    print(f"❌ 导入失败: {e}")
    import traceback
    traceback.print_exc()


✅ RBFKAN 导入成功, params=37,160


## Section 3: Data Loading & Initial Exploration


In [5]:
# ====== Core imports + column definitions + Scaler ======
import os, sys, copy, shutil, json, ast
sys.path.insert(0, r'c:\KAN-OLS-SHF\code\BSRBF_KAN-main')
import torch, torch.nn as nn, torch.optim as optim
import pandas as pd, numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import joblib

TARGET_COL = 'SHF__mW_m2_'
GEO_COLS = [
    'Magnetic_Anomaly_Value(nT)', 'DEM', 'RASTERVALU',
    'cpd', 'LAB', 'Thermal Conductivity （mW/K/m/）', 'H(km)', 'DTF',
    'sedthk-m', 'ro6', 'vp9', 'vs8',
    'DTR'
]
DIST_COLS = ['distance_vector_to_grids']

class StandardThenMinMaxScaler(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.ss = StandardScaler(); self.mm = MinMaxScaler()
    def fit(self, X, y=None):
        self.ss.fit(X); self.mm.fit(self.ss.transform(X)); return self
    def transform(self, X):
        return self.mm.transform(self.ss.transform(X))
    def inverse_transform(self, X):
        return self.ss.inverse_transform(self.mm.inverse_transform(X))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Imports OK | torch={torch.__version__[:4]} | device={device} | n_feat={len(GEO_COLS)}")


✅ Imports OK | torch=2.13 | device=cpu | n_feat=13


In [6]:
def load_raw_data(data_file):
    """Load raw data, supporting .xlsx and .pkl formats (no LAB_LAB operation)."""
    if data_file.endswith('.xlsx'):
        df = pd.read_excel(data_file)
    elif data_file.endswith('.pkl'):
        df = pd.read_pickle(data_file)
    else:
        raise ValueError("不支持的文件类型，请使用 .xlsx 或 .pkl")
    
    print("数据列:", df.columns.tolist())
    
    y = df[TARGET_COL].values.copy()
    X_geo = df[GEO_COLS].values.astype(np.float32)
    
    dist_col = df[DIST_COLS[0]]
    first_val = dist_col.iloc[0]
    if isinstance(first_val, str):
        X_dist = np.array([ast.literal_eval(s) for s in dist_col]).astype(np.float32)
    else:
        X_dist = np.array(dist_col.tolist()).astype(np.float32)
    
    return df, X_dist, X_geo, y

# Load data
df, X_dist_raw, X_geo_raw, y_raw = load_raw_data(args.data_file)
y_mean_global = np.mean(y_raw)

print(f"\n📊 数据概览:")
print(f"   样本数: {len(df)}")
print(f"   距离向量维度: {X_dist_raw.shape[1]}")
print(f"   地理特征维度: {X_geo_raw.shape[1]}")
print(f"   目标变量均值: {y_mean_global:.4f}")
print(f"   NaN in X_dist: {np.any(np.isnan(X_dist_raw))}, Inf: {np.any(np.isinf(X_dist_raw))}")
print(f"   NaN in X_geo:  {np.any(np.isnan(X_geo_raw))}, Inf: {np.any(np.isinf(X_geo_raw))}")


数据列: ['数据编号', 'Lon', 'Lat', 'SHF__mW_m2_', 'Magnetic_Anomaly_Value(nT)', 'DEM', 'RASTERVALU', 'cpd', 'LAB', 'Thermal Conductivity （mW/K/m/）', 'crsthk', 'sedthk-m', 'bd6', 'bd7', 'bd8', 'bd9', 'ro6', 'ro7', 'ro8', 'ro9', 'vp6', 'vp7', 'vp8', 'vp9', 'vs6', 'vs7', 'vs8', 'vs9', 'H(km)', 'DTF', 'DTR', 'distance_vector_to_grids']

📊 数据概览:
   样本数: 703
   距离向量维度: 625
   地理特征维度: 13
   目标变量均值: 66.3176
   NaN in X_dist: False, Inf: False
   NaN in X_geo:  False, Inf: False


## Section 4: Prediction Function & Model Construction


In [7]:
def create_model_optimizer_scheduler(args, input_dim, output_dim, device):
    """Build the model, optimizer, and learning-rate scheduler."""
    mt = args.model_type.lower()
    if mt == 'bsrbf_kan':
        model = BSRBF_KAN([input_dim, args.n_hidden, args.n_hidden, output_dim],
                          grid_size=args.grid_size, spline_order=args.spline_order)
    elif mt == 'mlp':
        model = MLP([input_dim, args.n_hidden, args.n_hidden, output_dim])
    elif mt == 'efficient_kan':
        model = EfficientKAN([input_dim, args.n_hidden, args.n_hidden, output_dim],
                             grid_size=args.grid_size, spline_order=args.spline_order)
    elif mt == 'faster_kan':
        model = FasterKAN([input_dim, args.n_hidden, args.n_hidden, output_dim], num_grids=args.num_grids)
    elif mt == 'fast_kan':
        model = FastKAN([input_dim, args.n_hidden, args.n_hidden, output_dim], num_grids=args.num_grids)
    elif mt == 'gottlieb_kan':
        model = GottliebKAN([input_dim, args.n_hidden, args.n_hidden, output_dim], spline_order=args.spline_order)
    else:
        raise ValueError(f"Unknown model_type: {mt}")
    
    model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    
    if args.scheduler_type == 'cosine':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    elif args.scheduler_type == 'exp':
        scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)
    elif args.scheduler_type == 'plateau':
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=20, min_lr=1e-6)
    else:
        raise ValueError(f"Unknown scheduler_type: {args.scheduler_type}")
    
    return model, optimizer, scheduler

print("✅ 预测函数与模型构建函数已定义")


✅ 预测函数与模型构建函数已定义


## Section 5: Training & Evaluation Helper Functions


In [8]:
def train_epoch(model, dataloader, optimizer, criterion, beta0, beta, device):
    """Train one epoch and return the average loss."""
    model.train()
    total_loss = 0.0
    for X_dist_b, X_geo_b, y_b in dataloader:
        X_dist_b = X_dist_b.to(device)
        X_geo_b = X_geo_b.to(device)
        y_b = y_b.to(device)
        optimizer.zero_grad()
        w_pred = model(X_dist_b)
        y_pred = _compute_y_pred(w_pred, X_geo_b, beta0, beta)
        loss = criterion(y_pred, y_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)


In [9]:
def evaluate_model(model, dataloader, beta0, beta, scaler_y, device, return_all=False, inverse=False):
    """Evaluate the model, returning MSE and R²."""
    model.eval()
    y_true_list, y_pred_list = [], []
    with torch.no_grad():
        for X_dist_b, X_geo_b, y_b in dataloader:
            X_dist_b = X_dist_b.to(device)
            X_geo_b = X_geo_b.to(device)
            w_pred = model(X_dist_b)
            y_pred_scaled = _compute_y_pred(w_pred, X_geo_b, beta0, beta)
            if inverse:
                y_pred_orig = scaler_y.inverse_transform(y_pred_scaled.cpu().numpy().reshape(-1,1)).flatten()
                y_true_orig = scaler_y.inverse_transform(y_b.cpu().numpy().reshape(-1,1)).flatten()
                y_true_list.extend(y_true_orig); y_pred_list.extend(y_pred_orig)
            else:
                y_true_list.extend(y_b.cpu().numpy().flatten())
                y_pred_list.extend(y_pred_scaled.cpu().numpy().flatten())
    yt, yp = np.array(y_true_list), np.array(y_pred_list)
    mse, r2 = mean_squared_error(yt, yp), r2_score(yt, yp)
    if return_all:
        return mse, r2, yt, yp
    return mse, r2

print("✅ evaluate_model 已定义")


✅ evaluate_model 已定义


## Section 6: Single-Fold Training Function (train_fold)


In [10]:
# ==================== Plot helpers ====================
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = ['Times New Roman', 'FangSong', 'SimHei', 'Microsoft YaHei']

def add_scale_bar_line(ax, segment_km=200, color='black', fontsize=12):
    xl, yl = ax.get_xlim(), ax.get_ylim(); w, h = xl[1]-xl[0], yl[1]-yl[0]
    seg = segment_km/(111.32*np.cos(np.radians(np.mean(yl))))
    xs = xl[0]+0.04*w; yb = yl[0]+0.04*h; lh = 0.01*h
    ax.hlines(y=yb, xmin=xs, xmax=xs+seg, colors=color, lw=1.5, clip_on=False)
    ax.vlines(x=xs, ymin=yb-lh/2, ymax=yb+lh/2, colors=color, lw=1.5, clip_on=False)
    ax.vlines(x=xs+seg, ymin=yb-lh/2, ymax=yb+lh/2, colors=color, lw=1.5, clip_on=False)
    ax.text(xs+seg/2, yb+lh/2+0.005*h, f'{segment_km} km', ha='center', va='bottom', fontsize=fontsize, color=color, clip_on=False)

def add_north_arrow(ax, sr=0.02, fs=12):
    xl, yl = ax.get_xlim(), ax.get_ylim(); w, h = xl[1]-xl[0], yl[1]-yl[0]
    x0 = xl[1]-0.06*w; y0 = yl[1]-0.06*h-0.02*h; r = sr*h; al = 0.8*r
    ax.add_patch(mpatches.Circle((x0,y0),r,fc='white',ec='black',lw=1,zorder=10,clip_on=False))
    ax.add_patch(mpatches.Polygon([(x0,y0+al),(x0-0.35*r,y0-al),(x0+0.35*r,y0-al)],fc='black',ec='none',zorder=11,clip_on=False))
    ax.text(x0,y0+r+0.01*h,'N',ha='center',va='bottom',fontsize=fs,fontweight='bold',color='black',clip_on=False)

def _compute_y_pred(wp, Xg, b0, b):
    return b0 + torch.sum(wp * b * Xg, dim=1)

def _evaluate(model, loader, b0, b, sy=None, yrv=None):
    model.eval(); ys, ts, yo = [], [], []
    with torch.no_grad():
        for xd, xg, yb in loader:
            yp = _compute_y_pred(model(xd), xg, b0, b)
            ys.extend(yp.cpu().numpy()); ts.extend(yb.cpu().numpy())
            if sy is not None and yrv is not None:
                yo.extend(sy.inverse_transform(yp.cpu().numpy().reshape(-1,1)).flatten())
    r2s = r2_score(ts, ys); mse = mean_squared_error(ts, ys)
    if sy is not None and yrv is not None:
        return r2s, r2_score(yrv, yo), np.array(yo), np.array(yrv), np.array(ys), mse
    return r2s, mse

def _train_epoch(model, opt, loader, b0, b, crit, mn):
    model.train(); tl = 0.0
    for xd, xg, yb in loader:
        opt.zero_grad(); loss = crit(_compute_y_pred(model(xd), xg, b0, b), yb)
        loss.backward()
        if mn > 0: torch.nn.utils.clip_grad_norm_(model.parameters(), mn)
        opt.step(); tl += loss.item()
    return tl / len(loader)

def train_fold(fold, train_idx, val_idx, X_dist_raw, X_geo_raw, y_raw, df, args, device, output_base):
    print(f"\n{'='*50}\nFold {fold}: Train={len(train_idx)}, Val={len(val_idx)}")
    fd = os.path.join(output_base, f'fold_{fold}'); Path(fd).mkdir(parents=True, exist_ok=True)
    tm = np.mean(y_raw[train_idx])

    nc = min(args.pca_components, X_dist_raw.shape[1])
    pca = PCA(n_components=nc)
    Xd_p = pca.fit_transform(X_dist_raw[train_idx]); Xv_p = pca.transform(X_dist_raw[val_idx])
    joblib.dump(pca, os.path.join(fd, 'pca.pkl'))

    sXd, sXg, sy = StandardThenMinMaxScaler(), StandardThenMinMaxScaler(), StandardThenMinMaxScaler()
    Xd = sXd.fit_transform(Xd_p); Xv = sXd.transform(Xv_p)
    Xg = sXg.fit_transform(X_geo_raw[train_idx]); Xgv = sXg.transform(X_geo_raw[val_idx])
    yt = sy.fit_transform(y_raw[train_idx].reshape(-1,1)).flatten()
    yv = sy.transform(y_raw[val_idx].reshape(-1,1)).flatten()
    for n, o in [('scaler_X_dist',sXd),('scaler_X_geo',sXg),('scaler_y',sy)]:
        joblib.dump(o, os.path.join(fd, f'{n}.pkl'))

    reg = Ridge(alpha=args.ridge_alpha); reg.fit(Xg, yt)
    b0 = torch.tensor(reg.intercept_, dtype=torch.float32, device=device)
    b = torch.tensor(reg.coef_, dtype=torch.float32, device=device)
    print(f"  Ridge R2={reg.score(Xg, yt):.4f}")
    np.savez(os.path.join(fd, 'reg_coefficients.npz'), beta0=reg.intercept_, beta=reg.coef_)

    idi, odo = Xd.shape[1], Xg.shape[1]
    mt = args.model_type.lower()
    if mt == 'rbf_kan':
        model = RBFKAN([idi, args.n_hidden, args.n_hidden, odo], num_rbf=args.num_rbf).to(device)
    elif mt == 'bsrbf_kan':
        model = BSRBF_KAN([idi, args.n_hidden, args.n_hidden, odo], grid_size=args.grid_size, spline_order=args.spline_order).to(device)
    elif mt == 'mlp':
        model = MLP([idi, args.n_hidden, args.n_hidden, odo]).to(device)
    else:
        raise ValueError(f"Unknown: {mt}")
    print(f"  Model={mt}  Params={sum(p.numel() for p in model.parameters()):,}  PCA={X_dist_raw.shape[1]}->{nc}")

    tr_ld = DataLoader(TensorDataset(torch.tensor(Xd,dtype=torch.float32,device=device),torch.tensor(Xg,dtype=torch.float32,device=device),torch.tensor(yt,dtype=torch.float32,device=device)),batch_size=args.batch_size,shuffle=True)
    vl_ld = DataLoader(TensorDataset(torch.tensor(Xv,dtype=torch.float32,device=device),torch.tensor(Xgv,dtype=torch.float32,device=device),torch.tensor(yv,dtype=torch.float32,device=device)),batch_size=args.batch_size,shuffle=False)
    tr_ns = DataLoader(TensorDataset(torch.tensor(Xd,dtype=torch.float32,device=device),torch.tensor(Xg,dtype=torch.float32,device=device),torch.tensor(yt,dtype=torch.float32,device=device)),batch_size=args.batch_size,shuffle=False)

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=20, factor=0.5) if args.scheduler_type=='plateau' else torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)
    crit = nn.HuberLoss(delta=args.huber_delta)
    bv, bs, be = -float('inf'), None, 0; p, pc = 80, 0

    # ===================== history.json (original scale, overwrite write) =====================
    hist_file = os.path.join(fd, 'history.json')
    hist = []
    tr_y_raw, val_y_raw = y_raw[train_idx], y_raw[val_idx]

    def _eval_orig(loader, raw_y):
        """Return (original R2, original MSE, original RMSE): computed after inverse-normalizing predictions."""
        _, r2o, yo, yt_o, _, _ = _evaluate(model, loader, b0, b, sy, yrv=raw_y)
        mse_o = float(mean_squared_error(yt_o, yo))
        return float(r2o), mse_o, float(np.sqrt(mse_o))

    def _write_history():
        """Overwrite entirely in 'w' mode so that retraining leaves no stale data."""
        with open(hist_file, 'w', encoding='utf-8') as f:
            json.dump(hist, f, indent=2, ensure_ascii=False)

    # ---- Epoch 0: baseline after Ridge fitting (original scale) ----
    tr_r2_o, tr_mse_o, tr_rmse_o = _eval_orig(tr_ns, tr_y_raw)
    vr_o, val_mse_o, val_rmse_o = _eval_orig(vl_ld, val_y_raw)
    hist.append({
        'ep': 0, 'tr_loss': None,
        'tr_mse': tr_mse_o, 'tr_rmse': tr_rmse_o, 'tr_r2': tr_r2_o,
        'val_mse': val_mse_o, 'val_rmse': val_rmse_o, 'val_r2': vr_o,
        'lr': args.lr,
    })
    _write_history()

    pbar = tqdm(range(1, args.epochs+1), desc=f"Fold {fold}", leave=False)
    for ep in pbar:
        cur_lr = opt.param_groups[0]['lr']
        tl = _train_epoch(model, opt, tr_ld, b0, b, crit, 0.5)
        sch.step(tl) if args.scheduler_type=='plateau' else sch.step()

        # Record pre-standardization MSE / RMSE / R² each epoch
        tr_r2_o, tr_mse_o, tr_rmse_o = _eval_orig(tr_ns, tr_y_raw)
        vr_o, val_mse_o, val_rmse_o = _eval_orig(vl_ld, val_y_raw)
        hist.append({
            'ep': ep, 'tr_loss': float(tl),
            'tr_mse': tr_mse_o, 'tr_rmse': tr_rmse_o, 'tr_r2': tr_r2_o,
            'val_mse': val_mse_o, 'val_rmse': val_rmse_o, 'val_r2': vr_o,
            'lr': float(cur_lr),
        })
        _write_history()  # overwrite each epoch so progress is visible even if training is interrupted

        if vr_o > bv: bv, bs, be, pc = vr_o, copy.deepcopy(model.state_dict()), ep, 0
        else: pc += 1
        pbar.set_postfix({'loss': f'{tl:.4f}', 'trRMSE': f'{tr_rmse_o:.2f}', 'valR2': f'{vr_o:.4f}', 'best': f'{bv:.4f}'})
        if pc >= p: print(f"  Early stop @ ep {ep}"); break
    # ============================================================================

    model.load_state_dict(bs); print(f"  Best: ep={be} valR2(orig)={bv:.4f}")
    tr_ev = DataLoader(TensorDataset(torch.tensor(Xd,dtype=torch.float32,device=device),torch.tensor(Xg,dtype=torch.float32,device=device),torch.tensor(yt,dtype=torch.float32,device=device)),batch_size=args.batch_size,shuffle=False)
    _, vr_o, yp_v, yt_v, _, val_mse_s = _evaluate(model, vl_ld, b0, b, sy, yrv=y_raw[val_idx])
    trs, tro, yp_t, yt_t, _, tr_mse_s = _evaluate(model, tr_ev, b0, b, sy, yrv=y_raw[train_idx])
    val_mse_o = mean_squared_error(yt_v, yp_v); val_rmse_o = np.sqrt(val_mse_o)
    tr_mse_o = mean_squared_error(yt_t, yp_t); tr_rmse_o = np.sqrt(tr_mse_o)
    print(f"  Final: TrR2(s)={trs:.4f} TrR2(o)={tro:.4f} TrMSE(o)={tr_mse_o:.2f} TrRMSE(o)={tr_rmse_o:.2f}")
    print(f"         ValR2(s)={bv:.4f} ValR2(o)={vr_o:.4f} ValMSE(o)={val_mse_o:.2f} ValRMSE(o)={val_rmse_o:.2f}")

    torch.save(bs, os.path.join(fd, 'best_val_model.pth'))
    cfg = {'model_type':mt,'n_hidden':args.n_hidden,'num_rbf':args.num_rbf,'pca':nc,'ridge_alpha':args.ridge_alpha,'lr':args.lr,'wd':args.weight_decay,'seed':args.seed}
    with open(os.path.join(fd, 'config.json'), 'w') as f: json.dump(cfg, f, indent=2)

    ai = np.concatenate([train_idx, val_idx])
    rdf = pd.DataFrame({'Set':['Train']*len(train_idx)+['Val']*len(val_idx),'True':np.concatenate([yt_t,yt_v]),'Predicted':np.concatenate([yp_t,yp_v])})
    rdf['Abs_Error']=np.abs(rdf['True']-rdf['Predicted']); rdf['Rel_Error_pct']=rdf['Abs_Error']/(rdf['True']+1e-6)*100
    if 'Lon' in df.columns: rdf['Lon']=df.loc[ai,'Lon'].values; rdf['Lat']=df.loc[ai,'Lat'].values
    rdf.to_csv(os.path.join(fd, f'fold_{fold}_predictions.csv'), index=False)

    return bv, vr_o, trs, tro, tm, val_mse_s, tr_mse_s, val_rmse_o, tr_rmse_o

print("train_fold ready")


train_fold ready


## Section 7: Run Cross-Validation Training


In [ ]:
# ---- Load this model's training data (adaptive mesh) ----
df, X_dist_raw, X_geo_raw, y_raw = load_raw_data(r'c:\KAN-OLS-SHF\output\model\mesh\shf_with_distance_vectors.pkl')

# ==================== Cell A: RBFKAN training (adaptive mesh) ====================
_rbf_args = type('Args',(),{})()
_rbf_args.data_file = r'c:\KAN-OLS-SHF\output\model\mesh\shf_with_distance_vectors.pkl'
_rbf_args.output_dir = r'c:\KAN-OLS-SHF\output\train\rbfkan'
_rbf_args.model_type = 'rbf_kan'; _rbf_args.n_hidden = 64; _rbf_args.num_rbf = 4
_rbf_args.grid_size = 15; _rbf_args.spline_order = 5; _rbf_args.epochs = 200
_rbf_args.lr = 1e-3; _rbf_args.weight_decay = 1e-3; _rbf_args.batch_size = 32
_rbf_args.pca_components = 20; _rbf_args.cv_folds = 10; _rbf_args.seed = 37
_rbf_args.scheduler_type = 'plateau'; _rbf_args.huber_delta = 1.0
_rbf_args.regressor = 'ridge'; _rbf_args.ridge_alpha = 10.0
print(f"\n{'='*60}\n🔬 Cell A: RBFKAN（自适应网格） | hidden={_rbf_args.n_hidden} rbf={_rbf_args.num_rbf} α={_rbf_args.ridge_alpha}\n{'='*60}")

output_base = _rbf_args.output_dir
Path(output_base).mkdir(parents=True, exist_ok=True)
kf = KFold(n_splits=_rbf_args.cv_folds, shuffle=True, random_state=_rbf_args.seed)
fold_results = []
for fold, (train_idx, val_idx) in enumerate(kf.split(X_dist_raw), 1):
    val_r2_s, val_r2_o, tr_r2_s, tr_r2_o, train_mean, val_mse_s, tr_mse_s, val_rmse_o, tr_rmse_o = train_fold(
        fold, train_idx, val_idx, X_dist_raw, X_geo_raw, y_raw, df, _rbf_args, device, output_base)
    fold_results.append({'fold':fold,'r2_scaled':val_r2_s,'r2_orig':val_r2_o,'train_r2_scaled':tr_r2_s,'train_r2_orig':tr_r2_o,'train_mean':train_mean,'val_rmse_orig':val_rmse_o,'val_mse_orig':val_rmse_o**2,'tr_rmse_orig':tr_rmse_o})

r2_s_vals=[r['r2_scaled'] for r in fold_results]; r2_o_vals=[r['r2_orig'] for r in fold_results]; tr_r2_s=[r['train_r2_scaled'] for r in fold_results]
print(f"\n{'='*75}")
print(f"📊 RBFKAN CV 汇总")
print(f"{'='*75}")
print(f"  {'Fold':<6} {'Val R²(sc)':>12} {'Val R²(orig)':>14} {'Val RMSE(orig)':>15} {'Tr R²(sc)':>12} {'TrainMean':>12}")
for r in fold_results:
    print(f"  {r['fold']:<6} {r['r2_scaled']:>12.4f} {r['r2_orig']:>14.4f} {r['val_rmse_orig']:>15.2f} {r['train_r2_scaled']:>12.4f} {r['train_mean']:>12.2f}")
print(f"\n  Mean Val R²: {np.mean(r2_s_vals):.4f} ± {np.std(r2_s_vals):.4f}  |  Best: {np.max(r2_s_vals):.4f} (Fold {np.argmax(r2_s_vals)+1})")

summary = {'params':{k:getattr(_rbf_args,k) for k in ['model_type','n_hidden','num_rbf','lr','weight_decay','ridge_alpha','epochs','seed']},
    'fold_results':[{k:(float(v) if isinstance(v,(np.floating,np.integer)) else v) for k,v in r.items()} for r in fold_results],
    'mean_val_r2':float(np.mean(r2_s_vals)),'best_fold':int(np.argmax(r2_s_vals)+1),'best_r2':float(np.max(r2_s_vals))}
with open(os.path.join(output_base,'cv_summary.json'),'w') as f: json.dump(summary,f,indent=2)

bfi=np.argmax(r2_s_vals); bf=fold_results[bfi]['fold']; bfd=os.path.join(output_base,f'fold_{bf}')
print(f"\n🏆 Best fold: {bf} (R²={r2_s_vals[bfi]:.4f})")
for fn in ['best_val_model.pth','pca.pkl','scaler_X_dist.pkl','scaler_X_geo.pkl','scaler_y.pkl','reg_coefficients.npz','config.json']:

    if os.path.exists(os.path.join(bfd,fn)): shutil.copy2(os.path.join(bfd,fn), os.path.join(output_base,fn))
print(f"✅ RBFKAN done: {output_base}")


数据列: ['数据编号', 'Lon', 'Lat', 'SHF__mW_m2_', 'Magnetic_Anomaly_Value(nT)', 'DEM', 'RASTERVALU', 'cpd', 'LAB', 'Thermal Conductivity （mW/K/m/）', 'crsthk', 'sedthk-m', 'bd6', 'bd7', 'bd8', 'bd9', 'ro6', 'ro7', 'ro8', 'ro9', 'vp6', 'vp7', 'vp8', 'vp9', 'vs6', 'vs7', 'vs8', 'vs9', 'H(km)', 'DTF', 'DTR', 'distance_vector_to_grids']

🔬 Cell A: RBFKAN（自适应网格） | hidden=64 rbf=4 α=10.0

Fold 1: Train=632, Val=71
  Ridge R2=0.0981
  Model=rbf_kan  Params=31,336  PCA=625->20


  Early stop @ ep 181
  Best: ep=101 valR2(orig)=0.3603
  Final: TrR2(s)=0.7633 TrR2(o)=0.7633 TrMSE(o)=37.62 TrRMSE(o)=6.13
         ValR2(s)=0.3603 ValR2(o)=0.3603 ValMSE(o)=168.20 ValRMSE(o)=12.97

Fold 2: Train=632, Val=71
  Ridge R2=0.1024
  Model=rbf_kan  Params=31,336  PCA=625->20


  Early stop @ ep 83
  Best: ep=3 valR2(orig)=0.1459
  Final: TrR2(s)=0.3388 TrR2(o)=0.3388 TrMSE(o)=115.99 TrRMSE(o)=10.77
         ValR2(s)=0.1459 ValR2(o)=0.1459 ValMSE(o)=99.04 ValRMSE(o)=9.95

Fold 3: Train=632, Val=71
  Ridge R2=0.0874
  Model=rbf_kan  Params=31,336  PCA=625->20


  Early stop @ ep 166
  Best: ep=86 valR2(orig)=0.6398
  Final: TrR2(s)=0.7344 TrR2(o)=0.7344 TrMSE(o)=45.13 TrRMSE(o)=6.72
         ValR2(s)=0.6398 ValR2(o)=0.6398 ValMSE(o)=59.34 ValRMSE(o)=7.70

Fold 4: Train=633, Val=70
  Ridge R2=0.1039
  Model=rbf_kan  Params=31,336  PCA=625->20


  Early stop @ ep 111
  Best: ep=31 valR2(orig)=0.2694
  Final: TrR2(s)=0.6372 TrR2(o)=0.6372 TrMSE(o)=58.13 TrRMSE(o)=7.62
         ValR2(s)=0.2694 ValR2(o)=0.2694 ValMSE(o)=182.31 ValRMSE(o)=13.50

Fold 5: Train=633, Val=70
  Ridge R2=0.1057
  Model=rbf_kan  Params=31,336  PCA=625->20


...

  Early stop @ ep 106
  Best: ep=26 valR2(orig)=0.1314
  Final: TrR2(s)=0.5678 TrR2(o)=0.5678 TrMSE(o)=76.63 TrRMSE(o)=8.75
         ValR2(s)=0.1314 ValR2(o)=0.1314 ValMSE(o)=83.20 ValRMSE(o)=9.12

Fold 7: Train=633, Val=70
  Ridge R2=0.0907
  Model=rbf_kan  Params=31,336  PCA=625->20


  Early stop @ ep 81
  Best: ep=1 valR2(orig)=0.2778
  Final: TrR2(s)=0.2699 TrR2(o)=0.2699 TrMSE(o)=128.04 TrRMSE(o)=11.32
         ValR2(s)=0.2778 ValR2(o)=0.2778 ValMSE(o)=83.50 ValRMSE(o)=9.14

Fold 8: Train=633, Val=70
  Ridge R2=0.0921
  Model=rbf_kan  Params=31,336  PCA=625->20


  Early stop @ ep 100
  Best: ep=20 valR2(orig)=0.2497
  Final: TrR2(s)=0.5283 TrR2(o)=0.5283 TrMSE(o)=77.68 TrRMSE(o)=8.81
         ValR2(s)=0.2497 ValR2(o)=0.2497 ValMSE(o)=158.89 ValRMSE(o)=12.61

Fold 9: Train=633, Val=70
  Ridge R2=0.1049
  Model=rbf_kan  Params=31,336  PCA=625->20


  Early stop @ ep 87
  Best: ep=7 valR2(orig)=0.2952
  Final: TrR2(s)=0.4161 TrR2(o)=0.4161 TrMSE(o)=99.68 TrRMSE(o)=9.98
         ValR2(s)=0.2952 ValR2(o)=0.2952 ValMSE(o)=109.65 ValRMSE(o)=10.47

Fold 10: Train=633, Val=70
  Ridge R2=0.0972
  Model=rbf_kan  Params=31,336  PCA=625->20


  Early stop @ ep 126
  Best: ep=46 valR2(orig)=0.4286
  Final: TrR2(s)=0.6297 TrR2(o)=0.6297 TrMSE(o)=62.46 TrRMSE(o)=7.90
         ValR2(s)=0.4286 ValR2(o)=0.4286 ValMSE(o)=99.51 ValRMSE(o)=9.98

📊 RBFKAN CV 汇总
  Fold     Val R²(sc)   Val R²(orig)  Val RMSE(orig)    Tr R²(sc)    TrainMean
  1            0.3603         0.3603           12.97       0.7633        66.30
  2            0.1459         0.1459            9.95       0.3388        66.33
  3            0.6398         0.6398            7.70       0.7344        66.38
  4            0.2694         0.2694           13.50       0.6372        66.13
  5            0.2904         0.2904            9.86       0.5963        66.22
  6            0.1314         0.1314            9.12       0.5678        66.48
  7            0.2778         0.2778            9.14       0.2699        66.29
  8            0.2497         0.2497           12.61       0.5283        66.41
  9            0.2952         0.2952           10.47       0.4161        66.

## Uniform Mesh Model Training (RBFKAN / BSRBF_KAN / MLP)

The following 3 training cells are from train_geothermal_normal.ipynb, using uniform mesh data (shf_with_distance_vectors_normal.pkl), output to the *_normal directories.


In [18]:
# ---- Load this model's training data (uniform mesh) ----
df, X_dist_raw, X_geo_raw, y_raw = load_raw_data(r'c:\KAN-OLS-SHF\output\model\mesh\shf_with_distance_vectors_normal.pkl')

# ==================== Cell B: RBFKAN training (uniform mesh) ====================
_rbf_args = type('Args',(),{})()
_rbf_args.data_file = r'c:\KAN-OLS-SHF\output\model\mesh\shf_with_distance_vectors_normal.pkl'
_rbf_args.output_dir = r'c:\KAN-OLS-SHF\output\train\rbfkan_normal'
_rbf_args.model_type = 'rbf_kan'; _rbf_args.n_hidden = 64; _rbf_args.num_rbf = 4
_rbf_args.grid_size = 15; _rbf_args.spline_order = 5; _rbf_args.epochs = 200
_rbf_args.lr = 1e-3; _rbf_args.weight_decay = 1e-3; _rbf_args.batch_size = 32
_rbf_args.pca_components = 20; _rbf_args.cv_folds = 10; _rbf_args.seed = 37
_rbf_args.scheduler_type = 'plateau'; _rbf_args.huber_delta = 1.0
_rbf_args.regressor = 'ridge'; _rbf_args.ridge_alpha = 10.0
print(f"\n{'='*60}\n🔬 Cell B: RBFKAN（均匀网格） | hidden={_rbf_args.n_hidden} rbf={_rbf_args.num_rbf} α={_rbf_args.ridge_alpha}\n{'='*60}")

output_base = _rbf_args.output_dir
Path(output_base).mkdir(parents=True, exist_ok=True)
kf = KFold(n_splits=_rbf_args.cv_folds, shuffle=True, random_state=_rbf_args.seed)
fold_results = []
for fold, (train_idx, val_idx) in enumerate(kf.split(X_dist_raw), 1):
    val_r2_s, val_r2_o, tr_r2_s, tr_r2_o, train_mean, val_mse_s, tr_mse_s, val_rmse_o, tr_rmse_o = train_fold(
        fold, train_idx, val_idx, X_dist_raw, X_geo_raw, y_raw, df, _rbf_args, device, output_base)
    fold_results.append({'fold':fold,'r2_scaled':val_r2_s,'r2_orig':val_r2_o,'train_r2_scaled':tr_r2_s,'train_r2_orig':tr_r2_o,'train_mean':train_mean,'val_rmse_orig':val_rmse_o,'val_mse_orig':val_rmse_o**2,'tr_rmse_orig':tr_rmse_o})

r2_s_vals=[r['r2_scaled'] for r in fold_results]; r2_o_vals=[r['r2_orig'] for r in fold_results]; tr_r2_s=[r['train_r2_scaled'] for r in fold_results]
print(f"\n{'='*75}")
print(f"📊 RBFKAN CV 汇总")
print(f"{'='*75}")
print(f"  {'Fold':<6} {'Val R²(sc)':>12} {'Val R²(orig)':>14} {'Val RMSE(orig)':>15} {'Tr R²(sc)':>12} {'TrainMean':>12}")
for r in fold_results:
    print(f"  {r['fold']:<6} {r['r2_scaled']:>12.4f} {r['r2_orig']:>14.4f} {r['val_rmse_orig']:>15.2f} {r['train_r2_scaled']:>12.4f} {r['train_mean']:>12.2f}")
print(f"\n  Mean Val R²: {np.mean(r2_s_vals):.4f} ± {np.std(r2_s_vals):.4f}  |  Best: {np.max(r2_s_vals):.4f} (Fold {np.argmax(r2_s_vals)+1})")

summary = {'params':{k:getattr(_rbf_args,k) for k in ['model_type','n_hidden','num_rbf','lr','weight_decay','ridge_alpha','epochs','seed']},
    'fold_results':[{k:(float(v) if isinstance(v,(np.floating,np.integer)) else v) for k,v in r.items()} for r in fold_results],
    'mean_val_r2':float(np.mean(r2_s_vals)),'best_fold':int(np.argmax(r2_s_vals)+1),'best_r2':float(np.max(r2_s_vals))}
with open(os.path.join(output_base,'cv_summary.json'),'w') as f: json.dump(summary,f,indent=2)

bfi=np.argmax(r2_s_vals); bf=fold_results[bfi]['fold']; bfd=os.path.join(output_base,f'fold_{bf}')
print(f"\n🏆 Best fold: {bf} (R²={r2_s_vals[bfi]:.4f})")
for fn in ['best_val_model.pth','pca.pkl','scaler_X_dist.pkl','scaler_X_geo.pkl','scaler_y.pkl','reg_coefficients.npz','config.json']:

    if os.path.exists(os.path.join(bfd,fn)): shutil.copy2(os.path.join(bfd,fn), os.path.join(output_base,fn))
print(f"✅ RBFKAN done: {output_base}")


数据列: ['数据编号', 'Lon', 'Lat', 'SHF__mW_m2_', 'Magnetic_Anomaly_Value(nT)', 'DEM', 'RASTERVALU', 'cpd', 'LAB', 'Thermal Conductivity （mW/K/m/）', 'crsthk', 'sedthk-m', 'bd6', 'bd7', 'bd8', 'bd9', 'ro6', 'ro7', 'ro8', 'ro9', 'vp6', 'vp7', 'vp8', 'vp9', 'vs6', 'vs7', 'vs8', 'vs9', 'H(km)', 'DTF', 'DTR', 'distance_vector_to_grids']

🔬 Cell B: RBFKAN（均匀网格） | hidden=64 rbf=4 α=10.0

Fold 1: Train=632, Val=71
  Ridge R2=0.0981
  Model=rbf_kan  Params=31,336  PCA=4131->20


  Early stop @ ep 112
  Best: ep=32 valR2(orig)=0.4299
  Final: TrR2(s)=0.6114 TrR2(o)=0.6114 TrMSE(o)=61.76 TrRMSE(o)=7.86
         ValR2(s)=0.4299 ValR2(o)=0.4299 ValMSE(o)=149.88 ValRMSE(o)=12.24

Fold 2: Train=632, Val=71
  Ridge R2=0.1024
  Model=rbf_kan  Params=31,336  PCA=4131->20


  Early stop @ ep 115
  Best: ep=35 valR2(orig)=0.2779
  Final: TrR2(s)=0.6109 TrR2(o)=0.6109 TrMSE(o)=68.25 TrRMSE(o)=8.26
         ValR2(s)=0.2779 ValR2(o)=0.2779 ValMSE(o)=83.73 ValRMSE(o)=9.15

Fold 3: Train=632, Val=71
  Ridge R2=0.0874
  Model=rbf_kan  Params=31,336  PCA=4131->20


  Early stop @ ep 135
  Best: ep=55 valR2(orig)=0.6181
  Final: TrR2(s)=0.6782 TrR2(o)=0.6782 TrMSE(o)=54.69 TrRMSE(o)=7.40
         ValR2(s)=0.6181 ValR2(o)=0.6181 ValMSE(o)=62.91 ValRMSE(o)=7.93

Fold 4: Train=633, Val=70
  Ridge R2=0.1039
  Model=rbf_kan  Params=31,336  PCA=4131->20


  Early stop @ ep 110
  Best: ep=30 valR2(orig)=0.2725
  Final: TrR2(s)=0.6320 TrR2(o)=0.6320 TrMSE(o)=58.95 TrRMSE(o)=7.68
         ValR2(s)=0.2725 ValR2(o)=0.2725 ValMSE(o)=181.53 ValRMSE(o)=13.47

Fold 5: Train=633, Val=70
  Ridge R2=0.1057
  Model=rbf_kan  Params=31,336  PCA=4131->20


...

  Early stop @ ep 108
  Best: ep=28 valR2(orig)=0.2061
  Final: TrR2(s)=0.5765 TrR2(o)=0.5765 TrMSE(o)=75.10 TrRMSE(o)=8.67
         ValR2(s)=0.2061 ValR2(o)=0.2061 ValMSE(o)=76.05 ValRMSE(o)=8.72

Fold 7: Train=633, Val=70
  Ridge R2=0.0907
  Model=rbf_kan  Params=31,336  PCA=4131->20


  Early stop @ ep 81
  Best: ep=1 valR2(orig)=0.2029
  Final: TrR2(s)=0.2343 TrR2(o)=0.2343 TrMSE(o)=134.28 TrRMSE(o)=11.59
         ValR2(s)=0.2029 ValR2(o)=0.2029 ValMSE(o)=92.16 ValRMSE(o)=9.60

Fold 8: Train=633, Val=70
  Ridge R2=0.0921
  Model=rbf_kan  Params=31,336  PCA=4131->20


  Early stop @ ep 96
  Best: ep=16 valR2(orig)=0.3022
  Final: TrR2(s)=0.5285 TrR2(o)=0.5285 TrMSE(o)=77.64 TrRMSE(o)=8.81
         ValR2(s)=0.3022 ValR2(o)=0.3022 ValMSE(o)=147.78 ValRMSE(o)=12.16

Fold 9: Train=633, Val=70
  Ridge R2=0.1049
  Model=rbf_kan  Params=31,336  PCA=4131->20


  Early stop @ ep 86
  Best: ep=6 valR2(orig)=0.3036
  Final: TrR2(s)=0.3933 TrR2(o)=0.3933 TrMSE(o)=103.57 TrRMSE(o)=10.18
         ValR2(s)=0.3036 ValR2(o)=0.3036 ValMSE(o)=108.35 ValRMSE(o)=10.41

Fold 10: Train=633, Val=70
  Ridge R2=0.0972
  Model=rbf_kan  Params=31,336  PCA=4131->20


  Early stop @ ep 114
  Best: ep=34 valR2(orig)=0.3739
  Final: TrR2(s)=0.6183 TrR2(o)=0.6183 TrMSE(o)=64.40 TrRMSE(o)=8.02
         ValR2(s)=0.3739 ValR2(o)=0.3739 ValMSE(o)=109.02 ValRMSE(o)=10.44

📊 RBFKAN CV 汇总
  Fold     Val R²(sc)   Val R²(orig)  Val RMSE(orig)    Tr R²(sc)    TrainMean
  1            0.4299         0.4299           12.24       0.6114        66.30
  2            0.2779         0.2779            9.15       0.6109        66.33
  3            0.6181         0.6181            7.93       0.6782        66.38
  4            0.2725         0.2725           13.47       0.6320        66.13
  5            0.2764         0.2764            9.95       0.6755        66.22
  6            0.2061         0.2061            8.72       0.5765        66.48
  7            0.2029         0.2029            9.60       0.2343        66.29
  8            0.3022         0.3022           12.16       0.5285        66.41
  9            0.3036         0.3036           10.41       0.3933        6

In [ ]:
# ---- Load this model's training data (uniform mesh) ----
df, X_dist_raw, X_geo_raw, y_raw = load_raw_data(r'c:\KAN-OLS-SHF\output\model\mesh\shf_with_distance_vectors_normal.pkl')

# ==================== Cell C: BSRBF_KAN training (uniform mesh) ====================
_bsrbf_args = type('Args',(),{})()
_bsrbf_args.data_file = r'c:\KAN-OLS-SHF\output\model\mesh\shf_with_distance_vectors_normal.pkl'
_bsrbf_args.output_dir = r'c:\KAN-OLS-SHF\output\train\bsrbf_kan_normal'
_bsrbf_args.model_type = 'bsrbf_kan'; _bsrbf_args.n_hidden = 64
_bsrbf_args.num_rbf = 4; _bsrbf_args.grid_size = 15; _bsrbf_args.spline_order = 5
_bsrbf_args.epochs = 200; _bsrbf_args.lr = 1e-3; _bsrbf_args.weight_decay = 1e-3
_bsrbf_args.batch_size = 32; _bsrbf_args.pca_components = 20; _bsrbf_args.cv_folds = 10
_bsrbf_args.seed = 37; _bsrbf_args.scheduler_type = 'plateau'; _bsrbf_args.huber_delta = 1.0
_bsrbf_args.regressor = 'ridge'; _bsrbf_args.ridge_alpha = 10.0
print(f"\n{'='*60}\n🔬 Cell C: BSRBF_KAN（均匀网格） | hidden={_bsrbf_args.n_hidden} grid={_bsrbf_args.grid_size} order={_bsrbf_args.spline_order} α={_bsrbf_args.ridge_alpha}\n{'='*60}")

output_base = _bsrbf_args.output_dir; Path(output_base).mkdir(parents=True, exist_ok=True)
kf = KFold(n_splits=_bsrbf_args.cv_folds, shuffle=True, random_state=_bsrbf_args.seed)
fold_results = []
for fold, (train_idx, val_idx) in enumerate(kf.split(X_dist_raw), 1):
    val_r2_s, val_r2_o, tr_r2_s, tr_r2_o, train_mean, val_mse_s, tr_mse_s, val_rmse_o, tr_rmse_o = train_fold(
        fold, train_idx, val_idx, X_dist_raw, X_geo_raw, y_raw, df, _bsrbf_args, device, output_base)
    fold_results.append({'fold':fold,'r2_scaled':val_r2_s,'r2_orig':val_r2_o,'train_r2_scaled':tr_r2_s,'train_r2_orig':tr_r2_o,'train_mean':train_mean,'val_rmse_orig':val_rmse_o,'val_mse_orig':val_rmse_o**2,'tr_rmse_orig':tr_rmse_o})

r2_s_vals=[r['r2_scaled'] for r in fold_results]; r2_o_vals=[r['r2_orig'] for r in fold_results]; tr_r2_s=[r['train_r2_scaled'] for r in fold_results]
print(f"\n{'='*75}")
print(f"📊 BSRBF_KAN CV 汇总")
print(f"{'='*75}")
print(f"  {'Fold':<6} {'Val R²(sc)':>12} {'Val R²(orig)':>14} {'Val RMSE(orig)':>15} {'Tr R²(sc)':>12} {'TrainMean':>12}")
for r in fold_results:
    print(f"  {r['fold']:<6} {r['r2_scaled']:>12.4f} {r['r2_orig']:>14.4f} {r['val_rmse_orig']:>15.2f} {r['train_r2_scaled']:>12.4f} {r['train_mean']:>12.2f}")
print(f"\n  Mean Val R²: {np.mean(r2_s_vals):.4f} ± {np.std(r2_s_vals):.4f}  |  Best: {np.max(r2_s_vals):.4f} (Fold {np.argmax(r2_s_vals)+1})")

summary = {'params':{k:getattr(_bsrbf_args,k) for k in ['model_type','n_hidden','grid_size','spline_order','lr','weight_decay','ridge_alpha','epochs','seed']},
    'fold_results':[{k:(float(v) if isinstance(v,(np.floating,np.integer)) else v) for k,v in r.items()} for r in fold_results],
    'mean_val_r2':float(np.mean(r2_s_vals)),'best_fold':int(np.argmax(r2_s_vals)+1),'best_r2':float(np.max(r2_s_vals))}
with open(os.path.join(output_base,'cv_summary.json'),'w') as f: json.dump(summary,f,indent=2)

bfi=np.argmax(r2_s_vals); bf=fold_results[bfi]['fold']; bfd=os.path.join(output_base,f'fold_{bf}')
print(f"\n🏆 Best fold: {bf} (R²={r2_s_vals[bfi]:.4f})")
for fn in ['best_val_model.pth','pca.pkl','scaler_X_dist.pkl','scaler_X_geo.pkl','scaler_y.pkl','reg_coefficients.npz','config.json']:

    if os.path.exists(os.path.join(bfd,fn)): shutil.copy2(os.path.join(bfd,fn), os.path.join(output_base,fn))
print(f"✅ BSRBF_KAN done: {output_base}")


数据列: ['数据编号', 'Lon', 'Lat', 'SHF__mW_m2_', 'Magnetic_Anomaly_Value(nT)', 'DEM', 'RASTERVALU', 'cpd', 'LAB', 'Thermal Conductivity （mW/K/m/）', 'crsthk', 'sedthk-m', 'bd6', 'bd7', 'bd8', 'bd9', 'ro6', 'ro7', 'ro8', 'ro9', 'vp6', 'vp7', 'vp8', 'vp9', 'vs6', 'vs7', 'vs8', 'vs9', 'H(km)', 'DTF', 'DTR', 'distance_vector_to_grids']

🔬 Cell C: BSRBF_KAN（均匀网格） | hidden=64 grid=15 order=5 α=10.0

Fold 1: Train=632, Val=71
  Ridge R2=0.0981
  Model=bsrbf_kan  Params=130,724  PCA=4131->20


  Early stop @ ep 112
  Best: ep=32 valR2(orig)=0.3328
  Final: TrR2(s)=0.9402 TrR2(o)=0.9402 TrMSE(o)=9.50 TrRMSE(o)=3.08
         ValR2(s)=0.3328 ValR2(o)=0.3328 ValMSE(o)=175.43 ValRMSE(o)=13.24

Fold 2: Train=632, Val=71
  Ridge R2=0.1024
  Model=bsrbf_kan  Params=130,724  PCA=4131->20


  Early stop @ ep 83
  Best: ep=3 valR2(orig)=0.2661
  Final: TrR2(s)=0.5854 TrR2(o)=0.5854 TrMSE(o)=72.73 TrRMSE(o)=8.53
         ValR2(s)=0.2661 ValR2(o)=0.2661 ValMSE(o)=85.11 ValRMSE(o)=9.23

Fold 3: Train=632, Val=71
  Ridge R2=0.0874
  Model=bsrbf_kan  Params=130,724  PCA=4131->20


  Early stop @ ep 95
  Best: ep=15 valR2(orig)=0.5444
  Final: TrR2(s)=0.8601 TrR2(o)=0.8601 TrMSE(o)=23.77 TrRMSE(o)=4.88
         ValR2(s)=0.5444 ValR2(o)=0.5444 ValMSE(o)=75.05 ValRMSE(o)=8.66

Fold 4: Train=633, Val=70
  Ridge R2=0.1039
  Model=bsrbf_kan  Params=130,724  PCA=4131->20


  Early stop @ ep 103
  Best: ep=23 valR2(orig)=0.2909
  Final: TrR2(s)=0.9102 TrR2(o)=0.9102 TrMSE(o)=14.39 TrRMSE(o)=3.79
         ValR2(s)=0.2909 ValR2(o)=0.2909 ValMSE(o)=176.96 ValRMSE(o)=13.30

Fold 5: Train=633, Val=70
  Ridge R2=0.1057
  Model=bsrbf_kan  Params=130,724  PCA=4131->20


...

  Early stop @ ep 90
  Best: ep=10 valR2(orig)=0.0992
  Final: TrR2(s)=0.8337 TrR2(o)=0.8337 TrMSE(o)=29.50 TrRMSE(o)=5.43
         ValR2(s)=0.0992 ValR2(o)=0.0992 ValMSE(o)=86.29 ValRMSE(o)=9.29

Fold 7: Train=633, Val=70
  Ridge R2=0.0907
  Model=bsrbf_kan  Params=130,724  PCA=4131->20


  Early stop @ ep 117
  Best: ep=37 valR2(orig)=0.3411
  Final: TrR2(s)=0.9558 TrR2(o)=0.9558 TrMSE(o)=7.75 TrRMSE(o)=2.78
         ValR2(s)=0.3411 ValR2(o)=0.3411 ValMSE(o)=76.18 ValRMSE(o)=8.73

Fold 8: Train=633, Val=70
  Ridge R2=0.0921
  Model=bsrbf_kan  Params=130,724  PCA=4131->20


  Early stop @ ep 83
  Best: ep=3 valR2(orig)=0.2427
  Final: TrR2(s)=0.6249 TrR2(o)=0.6249 TrMSE(o)=61.76 TrRMSE(o)=7.86
         ValR2(s)=0.2427 ValR2(o)=0.2427 ValMSE(o)=160.37 ValRMSE(o)=12.66

Fold 9: Train=633, Val=70
  Ridge R2=0.1049
  Model=bsrbf_kan  Params=130,724  PCA=4131->20


  Early stop @ ep 95
  Best: ep=15 valR2(orig)=0.3246
  Final: TrR2(s)=0.8595 TrR2(o)=0.8595 TrMSE(o)=23.99 TrRMSE(o)=4.90
         ValR2(s)=0.3246 ValR2(o)=0.3246 ValMSE(o)=105.08 ValRMSE(o)=10.25

Fold 10: Train=633, Val=70
  Ridge R2=0.0972
  Model=bsrbf_kan  Params=130,724  PCA=4131->20


  Early stop @ ep 103
  Best: ep=23 valR2(orig)=0.4152
  Final: TrR2(s)=0.9324 TrR2(o)=0.9324 TrMSE(o)=11.40 TrRMSE(o)=3.38
         ValR2(s)=0.4152 ValR2(o)=0.4152 ValMSE(o)=101.83 ValRMSE(o)=10.09

📊 BSRBF_KAN CV 汇总
  Fold     Val R²(sc)   Val R²(orig)  Val RMSE(orig)    Tr R²(sc)    TrainMean
  1            0.3328         0.3328           13.24       0.9402        66.30
  2            0.2661         0.2661            9.23       0.5854        66.33
  3            0.5444         0.5444            8.66       0.8601        66.38
  4            0.2909         0.2909           13.30       0.9102        66.13
  5            0.4275         0.4275            8.85       0.8618        66.22
  6            0.0992         0.0992            9.29       0.8337        66.48
  7            0.3411         0.3411            8.73       0.9558        66.29
  8            0.2427         0.2427           12.66       0.6249        66.41
  9            0.3246         0.3246           10.25       0.8595      

In [19]:
# ---- Load this model's training data (uniform mesh) ----
df, X_dist_raw, X_geo_raw, y_raw = load_raw_data(r'c:\KAN-OLS-SHF\output\model\mesh\shf_with_distance_vectors_normal.pkl')

# ==================== Cell D: MLP training (uniform mesh) ====================
_mlp_args = type('Args',(),{})()
_mlp_args.data_file = r'c:\KAN-OLS-SHF\output\model\mesh\shf_with_distance_vectors_normal.pkl'
_mlp_args.output_dir = r'c:\KAN-OLS-SHF\output\train\mlp_normal'
_mlp_args.model_type = 'mlp'; _mlp_args.n_hidden = 64
_mlp_args.num_rbf = 4; _mlp_args.grid_size = 15; _mlp_args.spline_order = 5
_mlp_args.epochs = 200; _mlp_args.lr = 1e-3; _mlp_args.weight_decay = 1e-3
_mlp_args.batch_size = 32; _mlp_args.pca_components = 20; _mlp_args.cv_folds = 10
_mlp_args.seed = 37; _mlp_args.scheduler_type = 'plateau'; _mlp_args.huber_delta = 1.0
_mlp_args.regressor = 'ridge'; _mlp_args.ridge_alpha = 10.0
print(f"\n{'='*60}\n🔬 Cell D: MLP（均匀网格） | hidden={_mlp_args.n_hidden} α={_mlp_args.ridge_alpha}\n{'='*60}")

output_base = _mlp_args.output_dir; Path(output_base).mkdir(parents=True, exist_ok=True)
kf = KFold(n_splits=_mlp_args.cv_folds, shuffle=True, random_state=_mlp_args.seed)
fold_results = []
for fold, (train_idx, val_idx) in enumerate(kf.split(X_dist_raw), 1):
    val_r2_s, val_r2_o, tr_r2_s, tr_r2_o, train_mean, val_mse_s, tr_mse_s, val_rmse_o, tr_rmse_o = train_fold(
        fold, train_idx, val_idx, X_dist_raw, X_geo_raw, y_raw, df, _mlp_args, device, output_base)
    fold_results.append({'fold':fold,'r2_scaled':val_r2_s,'r2_orig':val_r2_o,'train_r2_scaled':tr_r2_s,'train_r2_orig':tr_r2_o,'train_mean':train_mean,'val_rmse_orig':val_rmse_o,'val_mse_orig':val_rmse_o**2,'tr_rmse_orig':tr_rmse_o})

r2_s_vals=[r['r2_scaled'] for r in fold_results]; r2_o_vals=[r['r2_orig'] for r in fold_results]; tr_r2_s=[r['train_r2_scaled'] for r in fold_results]
print(f"\n{'='*75}")
print(f"📊 MLP CV 汇总")
print(f"{'='*75}")
print(f"  {'Fold':<6} {'Val R²(sc)':>12} {'Val R²(orig)':>14} {'Val RMSE(orig)':>15} {'Tr R²(sc)':>12} {'TrainMean':>12}")
for r in fold_results:
    print(f"  {r['fold']:<6} {r['r2_scaled']:>12.4f} {r['r2_orig']:>14.4f} {r['val_rmse_orig']:>15.2f} {r['train_r2_scaled']:>12.4f} {r['train_mean']:>12.2f}")
print(f"\n  Mean Val R²: {np.mean(r2_s_vals):.4f} ± {np.std(r2_s_vals):.4f}  |  Best: {np.max(r2_s_vals):.4f} (Fold {np.argmax(r2_s_vals)+1})")

summary = {'params':{k:getattr(_mlp_args,k) for k in ['model_type','n_hidden','lr','weight_decay','ridge_alpha','epochs','seed']},
    'fold_results':[{k:(float(v) if isinstance(v,(np.floating,np.integer)) else v) for k,v in r.items()} for r in fold_results],
    'mean_val_r2':float(np.mean(r2_s_vals)),'best_fold':int(np.argmax(r2_s_vals)+1),'best_r2':float(np.max(r2_s_vals))}
with open(os.path.join(output_base,'cv_summary.json'),'w') as f: json.dump(summary,f,indent=2)

bfi=np.argmax(r2_s_vals); bf=fold_results[bfi]['fold']; bfd=os.path.join(output_base,f'fold_{bf}')
print(f"\n🏆 Best fold: {bf} (R²={r2_s_vals[bfi]:.4f})")
for fn in ['best_val_model.pth','pca.pkl','scaler_X_dist.pkl','scaler_X_geo.pkl','scaler_y.pkl','reg_coefficients.npz','config.json']:

    if os.path.exists(os.path.join(bfd,fn)): shutil.copy2(os.path.join(bfd,fn), os.path.join(output_base,fn))
print(f"✅ MLP done: {output_base}")


数据列: ['数据编号', 'Lon', 'Lat', 'SHF__mW_m2_', 'Magnetic_Anomaly_Value(nT)', 'DEM', 'RASTERVALU', 'cpd', 'LAB', 'Thermal Conductivity （mW/K/m/）', 'crsthk', 'sedthk-m', 'bd6', 'bd7', 'bd8', 'bd9', 'ro6', 'ro7', 'ro8', 'ro9', 'vp6', 'vp7', 'vp8', 'vp9', 'vs6', 'vs7', 'vs8', 'vs9', 'H(km)', 'DTF', 'DTR', 'distance_vector_to_grids']

🔬 Cell D: MLP（均匀网格） | hidden=64 α=10.0

Fold 1: Train=632, Val=71
  Ridge R2=0.0981
  Model=mlp  Params=6,504  PCA=4131->20


  Best: ep=127 valR2(orig)=0.4317
  Final: TrR2(s)=0.6400 TrR2(o)=0.6400 TrMSE(o)=57.21 TrRMSE(o)=7.56
         ValR2(s)=0.4317 ValR2(o)=0.4317 ValMSE(o)=149.43 ValRMSE(o)=12.22

Fold 2: Train=632, Val=71
  Ridge R2=0.1024
  Model=mlp  Params=6,504  PCA=4131->20


  Early stop @ ep 151
  Best: ep=71 valR2(orig)=0.3209
  Final: TrR2(s)=0.6238 TrR2(o)=0.6238 TrMSE(o)=66.00 TrRMSE(o)=8.12
         ValR2(s)=0.3209 ValR2(o)=0.3209 ValMSE(o)=78.75 ValRMSE(o)=8.87

Fold 3: Train=632, Val=71
  Ridge R2=0.0874
  Model=mlp  Params=6,504  PCA=4131->20


  Best: ep=143 valR2(orig)=0.5935
  Final: TrR2(s)=0.6592 TrR2(o)=0.6592 TrMSE(o)=57.91 TrRMSE(o)=7.61
         ValR2(s)=0.5935 ValR2(o)=0.5935 ValMSE(o)=66.96 ValRMSE(o)=8.18

Fold 4: Train=633, Val=70
  Ridge R2=0.1039
  Model=mlp  Params=6,504  PCA=4131->20


  Best: ep=123 valR2(orig)=0.2669
  Final: TrR2(s)=0.6747 TrR2(o)=0.6747 TrMSE(o)=52.11 TrRMSE(o)=7.22
         ValR2(s)=0.2669 ValR2(o)=0.2669 ValMSE(o)=182.94 ValRMSE(o)=13.53

Fold 5: Train=633, Val=70
  Ridge R2=0.1057
  Model=mlp  Params=6,504  PCA=4131->20


...

  Early stop @ ep 114
  Best: ep=34 valR2(orig)=0.1519
  Final: TrR2(s)=0.5578 TrR2(o)=0.5578 TrMSE(o)=78.42 TrRMSE(o)=8.86
         ValR2(s)=0.1519 ValR2(o)=0.1519 ValMSE(o)=81.24 ValRMSE(o)=9.01

Fold 7: Train=633, Val=70
  Ridge R2=0.0907
  Model=mlp  Params=6,504  PCA=4131->20


  Early stop @ ep 161
  Best: ep=81 valR2(orig)=0.2323
  Final: TrR2(s)=0.6470 TrR2(o)=0.6470 TrMSE(o)=61.91 TrRMSE(o)=7.87
         ValR2(s)=0.2323 ValR2(o)=0.2323 ValMSE(o)=88.76 ValRMSE(o)=9.42

Fold 8: Train=633, Val=70
  Ridge R2=0.0921
  Model=mlp  Params=6,504  PCA=4131->20


  Early stop @ ep 126
  Best: ep=46 valR2(orig)=0.3347
  Final: TrR2(s)=0.5852 TrR2(o)=0.5852 TrMSE(o)=68.31 TrRMSE(o)=8.26
         ValR2(s)=0.3347 ValR2(o)=0.3347 ValMSE(o)=140.89 ValRMSE(o)=11.87

Fold 9: Train=633, Val=70
  Ridge R2=0.1049
  Model=mlp  Params=6,504  PCA=4131->20


  Early stop @ ep 119
  Best: ep=39 valR2(orig)=0.3236
  Final: TrR2(s)=0.5566 TrR2(o)=0.5566 TrMSE(o)=75.69 TrRMSE(o)=8.70
         ValR2(s)=0.3236 ValR2(o)=0.3236 ValMSE(o)=105.22 ValRMSE(o)=10.26

Fold 10: Train=633, Val=70
  Ridge R2=0.0972
  Model=mlp  Params=6,504  PCA=4131->20


  Early stop @ ep 95
  Best: ep=15 valR2(orig)=0.3491
  Final: TrR2(s)=0.4167 TrR2(o)=0.4167 TrMSE(o)=98.39 TrRMSE(o)=9.92
         ValR2(s)=0.3491 ValR2(o)=0.3491 ValMSE(o)=113.34 ValRMSE(o)=10.65

📊 MLP CV 汇总
  Fold     Val R²(sc)   Val R²(orig)  Val RMSE(orig)    Tr R²(sc)    TrainMean
  1            0.4317         0.4317           12.22       0.6400        66.30
  2            0.3209         0.3209            8.87       0.6238        66.33
  3            0.5935         0.5935            8.18       0.6592        66.38
  4            0.2669         0.2669           13.53       0.6747        66.13
  5            0.3844         0.3844            9.18       0.5050        66.22
  6            0.1519         0.1519            9.01       0.5578        66.48
  7            0.2323         0.2323            9.42       0.6470        66.29
  8            0.3347         0.3347           11.87       0.5852        66.41
  9            0.3236         0.3236           10.26       0.5566        66.48